# Notebook 3 — RAG e Análise de Erros

Este notebook cobre três partes:

1. **RAG** — indexação dos editais no ChromaDB e busca semântica com o Gemini
2. **Análise de erros** — identificação e categorização de falhas observadas no projeto
3. **Estimativa de custos** — projeção do custo de processar 1.000 editais em diferentes modelos Gemini

## 1. Imports

In [3]:
import os
import re
import sys
import json
import time
import requests
import chromadb
import numpy as np
import pandas as pd

from pathlib                 import Path
from dotenv                  import load_dotenv
from openai                  import OpenAI 
from pydantic                import ValidationError
from sentence_transformers   import SentenceTransformer
from sklearn.model_selection import train_test_split


# garante que o src/ do projeto está no path para importar o schema
sys.path.insert(0, str(Path("..").resolve()))
from src.schema import EditalClassificado



## 2. Configuração API

In [4]:
# Groq

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "Chave DEEPSEEK_API_KEY não encontrada no .env")

cliente = OpenAI( api_key=GROQ_API_KEY,
                  base_url="https://api.groq.com/openai/v1")

MODELO_PRINCIPAL = "llama-3.3-70b-versatile"

print(f"Modelo principal: {MODELO_PRINCIPAL}")
print("Cliente Groq configurado com sucesso.")


Modelo principal: llama-3.3-70b-versatile
Cliente Groq configurado com sucesso.


## 3. Carga dos Dados

In [6]:
# caminhos dos arquivos salvos nos notebooks anteriores

PROCESSED_PATH = Path("../dados/processed")
CHROMA_PATH    = PROCESSED_PATH / "chroma_db"
CHROMA_PATH.mkdir(parents=True, exist_ok=True)

print(f"API Groq configurada. Modelo: {MODELO_PRINCIPAL}")
print(f"Pasta ChromaDB: {CHROMA_PATH}")

API Gemini configurada. Modelo: llama-3.3-70b-versatile
Pasta ChromaDB: ../dados/processed/chroma_db


In [7]:
# carrega o dataframe processado no notebook 1
df = pd.read_parquet(PROCESSED_PATH / "editais_processado.parquet")

In [8]:

# garante que campos de metadados existem
# (alguns podem estar aninhados como dict — o notebook 1 extraiu uf e orgao)
if "uf" not in df.columns:
    df["uf"] = df["unidadeOrgao"].apply(lambda x: x.get("ufSigla") if isinstance(x, dict) else None)

if "orgao" not in df.columns:
    df["orgao"] = df["unidadeOrgao"].apply(lambda x: x.get("nomeUnidade") if isinstance(x, dict) else None)

In [9]:
# carrega os embeddings gerados no notebook 1
X_treino = np.load(PROCESSED_PATH / "X_treino.npy")
X_teste  = np.load(PROCESSED_PATH / "X_teste.npy")
y_treino = np.load(PROCESSED_PATH / "y_treino.npy", allow_pickle=True)
y_teste  = np.load(PROCESSED_PATH / "y_teste.npy",  allow_pickle=True)

print(f"DataFrame: {len(df)} editais")
print(f"Embeddings treino: {X_treino.shape}")
print(f"Embeddings teste:  {X_teste.shape}")
print(f"\nCategorias:\n{df['categoria'].value_counts()}")

DataFrame: 3009 editais
Embeddings treino: (2106, 384)
Embeddings teste:  (903, 384)

Categorias:
categoria
Pregão - Eletrônico          1899
Concorrência - Eletrônica    1060
Concorrência - Presencial      50
Name: count, dtype: int64


## 4. RAG

RAG (Retrieval-Augmented Generation) combina duas etapas:
1. **Busca**: encontra os editais mais parecidos com a pergunta do usuário
2. **Geração**: manda esses editais como contexto para o Gemini responder

Usamos dois corpora:
- **Histórico**: os 3.009 editais já coletados (indexados uma vez só)
- **Tempo real**: editais com propostas abertas agora (coletados a cada consulta)

### 4.1 Indexação do Corpus

In [11]:
# cria o banco vetorial persistente em disco
clienteChroma = chromadb.PersistentClient(path=str(CHROMA_PATH))

# cria (ou abre, se já existir) as duas coleções, usa similaridade de cosseno
colecaoHistorico = clienteChroma.get_or_create_collection(name="editais_historico", metadata={"hnsw:space": "cosine"})  

colecaoAbertos = clienteChroma.get_or_create_collection(name="editais_abertos", metadata={"hnsw:space": "cosine"})

print("Coleções criadas/abertas:")
print(f"  editais_historico: {colecaoHistorico.count()} documentos")
print(f"  editais_abertos:   {colecaoAbertos.count()} documentos")

Coleções criadas/abertas:
  editais_historico: 0 documentos
  editais_abertos:   0 documentos


In [ ]:
# recriando os mesmos splits para alihar o df com embeddings

df_filtrado = df[df["texto_limpo"] != ""].reset_index(drop=True)

df_treino, df_teste = train_test_split(
    df_filtrado,
    test_size=0.30,
    random_state=42,
    stratify=df_filtrado["categoria"]
)

# junta treino + teste para ter todos os 3.009 editais alinhados com embeddings
X_todos = np.vstack([X_treino, X_teste])
df_todos = pd.concat([df_treino, df_teste]).reset_index(drop=True)

print(f"Total de editais para indexar: {len(df_todos)}")
print(f"Shape dos embeddings: {X_todos.shape}")

In [ ]:
# só indexa se a coleção ainda estiver vazia (evita duplicar) 

if colecaoHistorico.count() == 0:
    print("Indexando corpus histórico no ChromaDB...")

    LOTE = 500  # insere 500 de cada vez para não sobrecarregar a memória

    for inicio in range(0, len(df_todos), LOTE):
        fim = min(inicio + LOTE, len(df_todos))

        lote_df  = df_todos.iloc[inicio:fim]
        lote_emb = X_todos[inicio:fim]

        ids        = [str(i) for i in range(inicio, fim)]
        embeddings = lote_emb.tolist()
        documentos = lote_df["texto_limpo"].fillna("").tolist()

        # metadados: apenas tipos simples (string, int, float)
        metadados = []
        for _, row in lote_df.iterrows():
            metadados.append({
                "categoria":      str(row.get("categoria", "") or ""),
                "modalidade":     str(row.get("modalidade_nome", "") or ""),
                "uf":             str(row.get("uf", "") or ""),
                "orgao":          str(row.get("orgao", "") or ""),
                "valor_estimado": float(row["valorTotalEstimado"])
                                  if pd.notna(row.get("valorTotalEstimado")) else 0.0})

        colecaoHistorico.add(ids=ids,
                             embeddings=embeddings,
                             documents=documentos,
                             metadatas=metadados)

        print(f"  Inserido lote {inicio}–{fim}")

    print(f"\nIndexação concluída: {colecaoHistorico.count()} documentos")

else:
    print(f"Coleção já indexada: {colecaoHistorico.count()} documentos. Pulando.")

### 4.2 Embeddings

In [ ]:
# carrega o mesmo modelo usado no notebook 1

modeloEmbedding = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

### 4.3 Coleta editais 

In [ ]:
def coletarEditaisAbertos(limite=100):
    """
    Busca editais com propostas ainda abertas na API do PNCP.
    Gera embeddings e indexa no ChromaDB na coleção editais_abertos.
    Sempre limpa a coleção antes de indexar (dados sempre frescos).
    """

    URL = "https://pncp.gov.br/api/consulta/v1/contratacoes/proposta"

    print("Coletando editais abertos da API...")

    editais = []
    pagina  = 1

    while len(editais) < limite:
        try:
            resposta = requests.get(
                URL,
                params={"pagina": pagina, "tamanhoPagina": 50},
                timeout=10
            )

            if resposta.status_code == 429:
                print("Rate limit. Aguardando 5s...")
                time.sleep(5)
                continue

            if resposta.status_code != 200:
                print(f"Erro HTTP {resposta.status_code}. Encerrando.")
                break

            dados    = resposta.json()
            registros = dados.get("data", [])

            if not registros:
                break

            editais.extend(registros)
            total_pag = dados.get("totalPaginas", 1)
            print(f"  Página {pagina}/{total_pag} — {len(registros)} editais")

            if pagina >= total_pag:
                break

            pagina += 1
            time.sleep(1.5)

        except Exception as e:
            print(f"Erro na coleta: {e}")
            break

    if not editais:
        print("Nenhum edital aberto encontrado.")
        return

    # monta os textos para embedding
    textos = [e.get("objetoCompra", "") or "" for e in editais]

    print(f"\nGerando embeddings para {len(textos)} editais abertos...")
    embeddings = modeloEmbedding.encode(textos, show_progress_bar=True)

    # limpa a coleção antes de reinserir (garante dados sempre atuais)
    colecaoAbertos.delete(ids=colecaoAbertos.get()["ids"])

    ids        = [f"aberto_{i}" for i in range(len(editais))]
    documentos = textos
    metadados  = []

    for e in editais:
        unidade = e.get("unidadeOrgao", {}) or {}
        metadados.append({
            "modalidade":     str(e.get("modalidadeNome", "") or ""),
            "uf":             str(unidade.get("ufSigla", "") or ""),
            "orgao":          str(unidade.get("nomeUnidade", "") or ""),
            "valor_estimado": float(e["valorTotalEstimado"])
                              if e.get("valorTotalEstimado") else 0.0,
        })

    colecaoAbertos.add(
        ids=ids,
        embeddings=embeddings.tolist(),
        documents=documentos,
        metadatas=metadados
    )

    print(f"Indexados {colecaoAbertos.count()} editais abertos na coleção.")

In [ ]:
def buscarEditaisSimilares(pergunta, colecao, n_resultados=5):
    """
    Gera embedding da pergunta e busca os editais mais similares no ChromaDB.
    Retorna lista com texto e metadados de cada resultado.
    """

    embedding_pergunta = modeloEmbedding.encode([pergunta])[0].tolist()

    resultados = colecao.query(
        query_embeddings=[embedding_pergunta],
        n_results=n_resultados,
        include=["documents", "metadatas", "distances"]
    )

    editais_encontrados = []

    for i in range(len(resultados["documents"][0])):
        editais_encontrados.append({
            "texto":     resultados["documents"][0][i],
            "metadados": resultados["metadatas"][0][i],
            "distancia": round(resultados["distances"][0][i], 4)
        })

    return editais_encontrados

In [ ]:
def responderComRAG(pergunta, colecao):
    """
    Recupera editais relevantes e usa o Gemini para gerar uma resposta
    fundamentada nesses editais.
    Retorna a resposta em texto e os editais usados como fonte.
    """

    editais = buscarEditaisSimilares(pergunta, colecao, n_resultados=5)

    # monta o contexto com os editais recuperados
    contexto = ""
    for i, e in enumerate(editais, 1):
        meta = e["metadados"]
        contexto += (
            f"\n--- Edital {i} ---\n"
            f"Texto: {e['texto']}\n"
            f"Modalidade: {meta.get('modalidade', 'não informado')}\n"
            f"UF: {meta.get('uf', 'não informado')}\n"
            f"Órgão: {meta.get('orgao', 'não informado')}\n"
            f"Valor estimado: R$ {meta.get('valor_estimado', 0):,.2f}\n"
        )

    prompt = (
        "Você é um assistente especializado em licitações públicas brasileiras.\n"
        "Com base SOMENTE nos editais abaixo, responda à pergunta do usuário.\n"
        "Se os editais não tiverem a informação, diga que não encontrou.\n\n"
        f"PERGUNTA: {pergunta}\n\n"
        f"EDITAIS RECUPERADOS:{contexto}\n\n"
        "RESPOSTA:"
    )

    try:
        resposta = cliente.models.generate_content(
            model=MODELO,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.2)
        )
        texto_resposta = resposta.text.strip()

    except Exception as e:
        texto_resposta = f"Erro ao chamar o Gemini: {e}"

    return {
        "pergunta":       pergunta,
        "resposta":       texto_resposta,
        "editais_fonte":  editais
    }

### 4.4 Teste RAG

In [ ]:
# teste simples para confirmar que o RAG está funcionando
pergunta_teste = "Quais os editais de serviços de saúde?"

print(f"Pergunta: {pergunta_teste}\n")

resultado_teste = responderComRAG(pergunta_teste, colecaoHistorico)

print("Resposta do Gemini:")
print(resultado_teste["resposta"])

print(f"\nEditais usados como fonte ({len(resultado_teste['editais_fonte'])}):")
for i, e in enumerate(resultado_teste["editais_fonte"], 1):
    print(f"  {i}. [{e['metadados'].get('categoria', '?')}] {e['texto'][:80]}...")